In [1]:
import numpy as np
import datetime
from train import train_model as tm
from predictor import predictor as pr
from cell_tracking import tracker as ct
import pandas as pd
from convert2CTCFormat import lineage_mask
from os.path import join
from multiprocessing import cpu_count
import hydra
from hydra.utils import to_absolute_path as abs_path
from hydra import compose, initialize
from omegaconf import OmegaConf

## Learn representation of cell movement

In [2]:
now = datetime.datetime.now()
print('START!', now)
with initialize(version_base='1.3', config_path="config"):
    cfg = compose(config_name="tracker")
    tm(cfg)
    now = datetime.datetime.now()
    print('train DONE!', now)

START! 2026-09-07 17:09:27.097303
please check the configures:

path: ''
dataloader:
  division_detect: true
  start_frame: 0
  num_frame: 200
  itv: 1
  if_crop: false
  tile_num: 25
  pred_tile_num: 4
  overlap: 32
train:
  epochs: 100
  batch_size: 2
  lr: 0.0001
  train_load: false
  load: CP.pth
  val: false
track:
  max_movenment: 50
  load_track: false
  track_file: track_linear_solver.csv
  centroid_file: 2025-09-22-centroid.npy
  method: linear_solver
  run_num: 1
  last_itv: 20
  division: true
  post_pro: true
  min_length: 1
  prune_leaf: true
  merge: true
  jitter_thr: 0.6
  div_interval: 30
  nearest: false



INFO: Using device cuda
INFO: Creating dataset with 5 examples
INFO: Starting training:
        Epochs:          100
        Batch size:      2
        Learning rate:   0.0001
        Training size:   4
        Checkpoints:     C:\Users\Min Lab\LED\result\checkpoint
        Device:          cuda
        Interval:        1
        Optimizer:       Adam
    
Epoch 1/100: 100%|███████████████████████████████| 4/4 [00:19<00:00,  4.93s/img, loss (batch)=1.13e+3]
INFO: Checkpoint 1 saved !
Epoch 2/100: 100%|███████████████████████████████████| 4/4 [00:20<00:00,  5.14s/img, loss (batch)=659]
INFO: Checkpoint 2 saved !
Epoch 3/100: 100%|███████████████████████████████████| 4/4 [00:22<00:00,  5.51s/img, loss (batch)=296]
INFO: Checkpoint 3 saved !
Epoch 4/100: 100%|███████████████████████████████████| 4/4 [00:15<00:00,  4.00s/img, loss (batch)=192]
INFO: Checkpoint 4 saved !
Epoch 5/100: 100%|███████████████████████████████████| 4/4 [00:16<00:00,  4.08s/img, loss (batch)=155]
INFO: Checkpoint 5

train DONE! 2026-09-07 17:38:52.822362


## predict movement field

In [3]:
now = datetime.datetime.now()
print('START!', now)
with initialize(version_base='1.3', config_path="config"):
    cfg = compose(config_name="tracker")
    pr(cfg)
    now = datetime.datetime.now()
    print('predict DONE!', now)

START! 2026-09-07 17:38:52.839965


INFO: Using device cuda


please check the configures:

path: ''
dataloader:
  division_detect: true
  start_frame: 0
  num_frame: 200
  itv: 1
  if_crop: false
  tile_num: 25
  pred_tile_num: 4
  overlap: 32
train:
  epochs: 100
  batch_size: 2
  lr: 0.0001
  train_load: false
  load: CP.pth
  val: false
track:
  max_movenment: 50
  load_track: false
  track_file: track_linear_solver.csv
  centroid_file: 2025-09-22-centroid.npy
  method: linear_solver
  run_num: 1
  last_itv: 20
  division: true
  post_pro: true
  min_length: 1
  prune_leaf: true
  merge: true
  jitter_thr: 0.6
  div_interval: 30
  nearest: false



INFO: Model loaded from C:\Users\Min Lab\LED\result\checkpoint\CP_epoch99.pth
INFO: Creating dataset with 5 examples
                                                                                                      

predict DONE! 2026-09-07 17:39:05.079961


## Bayesian Estimation and cell tracking

In [4]:
now = datetime.datetime.now()
print('START!', now)
with initialize(version_base='1.3', config_path="config"):
    cfg = compose(config_name="tracker")
    ct(cfg)
    now = datetime.datetime.now()
    print('track DONE!', now)

START! 2026-09-07 17:39:05.121707
please check the configures:

path: ''
dataloader:
  division_detect: true
  start_frame: 0
  num_frame: 200
  itv: 1
  if_crop: false
  tile_num: 25
  pred_tile_num: 4
  overlap: 32
train:
  epochs: 100
  batch_size: 2
  lr: 0.0001
  train_load: false
  load: CP.pth
  val: false
track:
  max_movenment: 50
  load_track: false
  track_file: track_linear_solver.csv
  centroid_file: 2025-09-22-centroid.npy
  method: linear_solver
  run_num: 1
  last_itv: 20
  division: true
  post_pro: true
  min_length: 1
  prune_leaf: true
  merge: true
  jitter_thr: 0.6
  div_interval: 30
  nearest: false

------0th running start-------

queue size: 1/1 ------0th running end-------

------all running ended-------

2026-09-07 17:09:265-frame time cost: 3 s
track DONE! 2026-09-07 17:39:09.119706


## save as CTC format and a gif file

In [5]:
mask_dir = abs_path(join('data', 'mask'))
track_dir = abs_path(join("result", "track_results.csv"))
centroid = abs_path(join("result", "centroid.npy"))

tracks = pd.read_csv(track_dir).to_numpy()
cnt = np.load(centroid, allow_pickle=True)
lineage_mask(mask_dir, tracks, cnt, save_path=r"result", res_path='RES1')
lineage_mask(mask_dir, tracks, cnt, save_path=r"result", res_path='RES1', saveRGB=True)

track_mask.gif saved!
